In [1]:
import json
import logging
from pathlib import Path
from typing import Any, Dict

import requests

In [2]:
def save_json(obj, path: str):
    p = Path(path)
    p.parent.mkdir(parents=True, exist_ok=True)
    with p.open("w", encoding="utf-8") as f:
        json.dump(obj, f, indent=2, ensure_ascii=False)
    return str(p)


def load_json(path: str):
    with open(path, "r", encoding="utf-8") as f:
        return json.load(f)


def load_connector_json(file_path: str) -> Dict[str, Any]:
    path = Path(file_path)
    if not path.exists():
        raise FileNotFoundError(f"File not found: {path}")
    with path.open("r", encoding="utf-8") as f:
        return json.load(f)


def create_connector(url: str, config: dict):
    r = requests.post(f"{url}/connectors", headers={"Content-Type": "application/json"}, data=json.dumps(config), timeout=10)
    logging.info(f"status code: {r.status_code}")
    return r.text


def status_connector(url: str, name: str):
    r = requests.get(f"{url}/connectors/{name}/status", timeout=10)
    logging.info(f"status code: {r.status_code}")
    return r.json() if r.headers.get("Content-Type", "").startswith("application/json") else r.text


def delete_connector(url: str, name: str):
    r = requests.delete(f"{url}/connectors/{name}", timeout=10)
    logging.info(f"status code: {r.status_code}")
    return r


def upsert_connector(url: str, config: dict):
    name = config["name"]
    r = requests.get(f"{url}/connectors/{name}", timeout=10)
    if r.status_code == 200:
        r2 = requests.put(f"{url}/connectors/{name}/config", headers={"Content-Type": "application/json"}, data=json.dumps(config), timeout=10)
        return r2.status_code, r2.text
    elif r.status_code == 404:
        return create_connector(url, config)
    else:
        return r.status_code, r.text

In [3]:
_kafka_connect_url = "http://confluent-kafka-connect:8083"

## Debezium Mysql CDC Connector (Source)

In [4]:
mysql_config = load_json("./config/mysql-mmix-source-connector.json")
create_connector(_kafka_connect_url, mysql_config)
#status_connector(_kafka_connect_url, mysql_config["name"])
#delete_connector(_kafka_connect_url, mysql_config["name"])

'{"error_code":409,"message":"Connector mysql-source-connector already exists"}'

## Confluent Kafka Connector S3 등록 (Sync)

In [37]:
s3_config = load_json("./config/s3-mmix-sync-connector.json")
create_connector(_kafka_connect_url, s3_config)
#status_connector(_kafka_connect_url, s3_config["name"])
#delete_connector(_kafka_connect_url, s3_config["name"])

'{"name":"s3-sync-connector","config":{"tasks.max":"1","connector.class":"io.confluent.connect.s3.S3SinkConnector","flush.size":"10000","rotate.interval.ms":"600000","partition.duration.ms":"3600000","locale":"ko_KR","timezone":"UTC","topics.dir":"messaging/mmix","topics.regex":"debezium\\\\.mysql\\\\.source\\\\.mmix\\\\..*","partitioner.class":"io.confluent.connect.storage.partitioner.TimeBasedPartitioner","format.class":"io.confluent.connect.s3.format.parquet.ParquetFormat","parquet.codec":"zstd","key.converter":"org.apache.kafka.connect.storage.StringConverter","value.converter":"io.confluent.connect.avro.AvroConverter","value.converter.schema.registry.url":"http://confluent-schema-registry:8081","value.converter.schemas.enable":"true","path.format":"\'year\'=YYYY/\'month\'=MM/\'day\'=dd/\'hour\'=HH","storage.class":"io.confluent.connect.s3.storage.S3Storage","store.url":"http://minio:9000","aws.access.key.id":"mmix","aws.secret.access.key":"mmixmmix","s3.bucket.name":"mmix-prod-dat